# Lab 03｜通勤時間分布 Distributions

<a href="https://colab.research.google.com/github/johnnychao/statistics-in-context-bilingual/blob/main/labs/colab/lab-03-distributions.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

> Statistics in Context · Unit 1 · 原創合成資料 · 不評量 Python 語法


## Goal

情境：晨間活動應該幾點開始，才不會忽略通勤較久的學生？

- 用 dot/strip plot 與 histogram 探索數值變數。
- 以 **shape, center, variability, unusual features** 描述分布。
- 說明遺漏值如何被排除，並比較不同 bin width 的影響。


## Setup

依序執行儲存格即可，不需要撰寫或背誦 Python。若想重新開始，請在 Colab 選擇 **Runtime → Restart session and run all**。

本 Lab 使用原創合成資料；所有代碼與數值均不對應真實學生。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


In [ ]:
# 集中設定：一般情況只需修改這一格的參數。
DATA_RELATIVE_PATH = "data/public/morning_routine_survey.csv"
REPO_RAW_BASE_URL = "https://raw.githubusercontent.com/johnnychao/statistics-in-context-bilingual/main"

LOCAL_REPO_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/content/statistics-in-context-bilingual"),
]


def load_repo_csv(relative_path):
    # 先找本機 repo，再讀 GitHub raw；失敗時提供繁中修復訊息。
    relative_path = Path(relative_path)
    for candidate_root in LOCAL_REPO_ROOT_CANDIDATES:
        candidate = candidate_root / relative_path
        if candidate.is_file():
            return pd.read_csv(candidate), str(candidate.resolve())

    remote_url = f"{REPO_RAW_BASE_URL}/{relative_path.as_posix()}"
    try:
        return pd.read_csv(remote_url), remote_url
    except Exception as exc:
        raise RuntimeError(
            "無法載入資料。請確認網路連線，或從 GitHub repo 根目錄執行此 Notebook。"
            f" 嘗試的遠端網址：{remote_url}。"
            " 若 repo 尚未發布，請先將 data/public 的 CSV 上傳至 main branch。"
        ) from exc


data, data_source = load_repo_csv(DATA_RELATIVE_PATH)
print(f"已載入 {len(data)} 筆資料｜Loaded {len(data)} rows")
print(f"來源 Source: {data_source}")


## Steps

### 1. 清理一個數值變數

先將空白值排除。`valid_values` 的筆數會少於 240，這是資料字典已聲明的合成 item nonresponse。


In [ ]:
VARIABLE = "commute_minutes"
valid_values = data[VARIABLE].dropna()

print(f"有效筆數 Valid n = {len(valid_values)}")
print(f"遺漏筆數 Missing n = {data[VARIABLE].isna().sum()}")
display(valid_values.describe().to_frame(name=VARIABLE))


### 2. 比較 strip plot 與 histogram

`BIN_WIDTH` 只改變圖的分組方式，不會改變原始資料。請先執行 10 分鐘，再改成 5 或 15 分鐘。


In [ ]:
# ✏️ 修改任務：比較 BIN_WIDTH = 5、10、15 時對形狀判讀的影響。
BIN_WIDTH = 10
if BIN_WIDTH <= 0:
    raise ValueError("BIN_WIDTH 必須大於 0。")

FIGURE_ALT = (
    "A strip plot and histogram of one-way commute minutes for synthetic students; "
    "most values are in the lower and middle ranges with a tail toward longer commutes."
)

bin_edges = np.arange(0, valid_values.max() + BIN_WIDTH * 2, BIN_WIDTH)
fig, axes = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={"height_ratios": [1, 3]})
sns.stripplot(x=valid_values, jitter=0.22, size=4, color="#E76F51", ax=axes[0])
axes[0].set_title("Individual commute times (synthetic survey)")
axes[0].set_xlabel("One-way commute time（minutes）")
sns.histplot(valid_values, bins=bin_edges, color="#2A9D8F", edgecolor="white", ax=axes[1])
axes[1].set_title(f"Distribution of commute time — bin width {BIN_WIDTH} minutes")
axes[1].set_xlabel("One-way commute time（minutes）")
axes[1].set_ylabel("Number of synthetic students（人數）")
plt.tight_layout()
plt.show()
print(f"Alt text: {FIGURE_ALT}")


### 3. 取得描述分布所需的證據

IQR rule 將低於 `Q1 − 1.5×IQR` 或高於 `Q3 + 1.5×IQR` 的值標記為 potential outliers。它是規則，不等於資料錯誤。


In [ ]:
q1 = valid_values.quantile(0.25)
median = valid_values.median()
q3 = valid_values.quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
potential_outliers = valid_values[(valid_values < lower_fence) | (valid_values > upper_fence)]

print(f"Median = {median:.1f} minutes")
print(f"IQR = {iqr:.1f} minutes (Q1={q1:.1f}, Q3={q3:.1f})")
print(f"Skewness = {valid_values.skew():.2f}")
print(f"Potential outliers above {upper_fence:.1f}: {sorted(potential_outliers.tolist())}")


<details>
<summary><strong>AP English Response frame</strong></summary>

> The distribution of commute time is [shape], with a median of about ____ minutes and an IQR of about ____ minutes. Most synthetic students commute between ____ and ____ minutes. A possible unusual feature is ____. In context, this means ____.

</details>

請勿只寫「右偏、有異常值」；至少加入一個 center、一個 variability 數值與通勤情境。


## Checks

檢查有效筆數與五數摘要的順序。


In [ ]:
assert data[VARIABLE].isna().sum() == 6
assert valid_values.min() <= q1 <= median <= q3 <= valid_values.max()
assert iqr >= 0
print("✅ Checks passed：遺漏值與五數摘要順序合理。")


## Next Steps

Lab 04 將比較 mean、median、SD、IQR，並用 boxplot 比較有無吃早餐的學生群組。先預測：哪一個群組的 readiness_score 中位數可能較高？
